## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 💬 Azure AI Agent with Thread Management - Multi-Turn Conversations 🏦

This notebook demonstrates working with conversation threads using `FoundryAgent` for multi-turn conversations like loan application discussions and financial planning sessions.

## Features Covered:
- Creating and managing persistent conversation threads
- Using `agent.get_new_thread()` for thread management
- Thread lifecycle management (create, use, delete)
- Multi-turn conversation continuity for banking discussions
- Thread initialization and validation with `store=False` option

### ⚠️ Important Note ⚠️
> **Threads maintain conversation context across multiple interactions. This is essential for complex banking discussions like loan applications or investment planning.**

## Prerequisites

Before running this notebook, ensure you have:

1. **Azure AI Project**: Access to an Microsoft Foundry project with deployed models
2. **Authentication**: Azure CLI installed and authenticated (`az login --use-device-code`)
3. **Environment Variables**: Set up your `.env` file with:
   - `AI_FOUNDRY_PROJECT_ENDPOINT`
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME`
4. **Dependencies**: Required agent-framework packages installed

If you need to use a different tenant:
```bash
az login --tenant <tenant-id>
```

This example demonstrates how to work with existing threads to maintain conversation continuity in banking scenarios.

## Import Libraries

Import the required libraries using the `Agent` + `FoundryChatClient` pattern:


In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import asyncio
import os
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Annotated

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from pydantic import Field

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
load_dotenv(repo_root / ".env", override=False)

endpoint = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
model = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not endpoint or not model:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")

print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print({package: version(package) for package in ("agent-framework-core", "agent-framework-foundry", "azure-ai-projects")})
print("Project endpoint and model: configured (values hidden)")


## Check Environment Variables

Verify that the required environment variables are set:

In [ ]:
if not endpoint or not model:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")
print("Project endpoint and model: configured (values hidden)")


## Define Function Tools 🏦

Let's define banking functions for our loan application discussions:

In [ ]:
def get_loan_prequalification(
    loan_amount: Annotated[float, Field(description="Requested loan amount in dollars.")],
    credit_score: Annotated[int, Field(description="Customer's credit score.")],
    annual_income: Annotated[float, Field(description="Customer's annual income in dollars.")],
) -> str:
    """Check loan prequalification status based on customer data."""
    # Simple prequalification logic for demo
    dti = loan_amount / (annual_income * 30)  # Simplified debt-to-income estimate
    
    if credit_score >= 700 and dti < 0.43:
        status = "Pre-Qualified ✅"
        rate = "5.99% - 6.75%"
    elif credit_score >= 650 and dti < 0.50:
        status = "Conditionally Pre-Qualified ⚠️"
        rate = "7.25% - 8.50%"
    else:
        status = "Additional Review Required 📋"
        rate = "8.50% - 12.00%"
    
    return f"""Loan Pre-Qualification Result:
    Requested Amount: ${loan_amount:,.2f}
    Credit Score: {credit_score}
    Annual Income: ${annual_income:,.2f}
    Status: {status}
    Estimated Rate Range: {rate}"""


def get_required_documents(
    loan_type: Annotated[str, Field(description="Type of loan: mortgage, auto, personal")],
) -> str:
    """Get list of required documents for loan application."""
    docs = {
        "mortgage": [
            "Government-issued ID",
            "Pay stubs (last 30 days)",
            "W-2 forms (last 2 years)",
            "Tax returns (last 2 years)",
            "Bank statements (last 2 months)",
            "Proof of assets",
            "Property information"
        ],
        "auto": [
            "Government-issued ID",
            "Proof of income",
            "Proof of insurance",
            "Vehicle information (VIN, mileage)"
        ],
        "personal": [
            "Government-issued ID",
            "Proof of income",
            "Proof of residence"
        ]
    }
    loan_type = loan_type.lower()
    if loan_type in docs:
        doc_list = "\n    - ".join(docs[loan_type])
        return f"Required Documents for {loan_type.title()} Loan:\n    - {doc_list}"
    return "Please specify: mortgage, auto, or personal loan"

## Create and Resume a Persistent Session 💬

This example shows how to:
1. Create a session that persists for the conversation with `agent.create_session()`
2. Resume an existing server-side session with `agent.get_session(service_session_id)`
3. Continue a multi-turn discussion with full context

**Key API Methods**:
- `agent.create_session()` — starts a new session; the service assigns `session.service_session_id` after the first run
- `agent.get_session(service_session_id)` — resumes an existing server-side session
- `agent.run(query, session=session)` — runs a turn within a session


In [ ]:
async def main() -> None:
    """Start a session, then resume it by its service session id to prove continuity."""
    print("=== 💬 Foundry Agent with Persistent Session - Loan Application ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="LoanApplicationAdvisor",
                instructions="""You are a Loan Application Advisor. Help customers through the loan application process.
                Remember context from earlier in the conversation to provide personalized guidance.
                Be professional and thorough in explaining requirements.""",
                tools=[get_loan_prequalification, get_required_documents],
            )

            # First turn creates a server-side session/thread.
            session = agent.create_session()
            query = (
                "I'm interested in applying for a mortgage. Can you check if I'd prequalify for $350,000 "
                "with a credit score of 720 and annual income of $95,000?"
            )
            print(f"\n🤔 Customer: {query}")
            async with asyncio.timeout(90):
                result = await agent.run(query, session=session)
            assert result.text, "The service returned no answer."
            print(f"🏦 Advisor: {result.text}")

            service_session_id = session.service_session_id
            print(f"\n🔗 Service session id: {'captured' if service_session_id else 'not provided by service'}")

            # Resume the existing server-side session to continue with full context.
            resumed = agent.get_session(service_session_id) if service_session_id else session
            follow_up = "Remind me of the loan amount and credit score I just gave you, then list the documents I'll need."
            print(f"\n🤔 Customer: {follow_up}")
            async with asyncio.timeout(90):
                result = await agent.run(follow_up, session=resumed)
            assert result.text, "The service returned no answer."
            print(f"🏦 Advisor: {result.text}")
        finally:
            await client.client.close()
            await client.project_client.close()


## 🚀 Execute the Initial Loan Discussion

In [ ]:
await main()

## 🔄 Multi-Turn Loan Application Flow with Thread Continuity

In [ ]:
async def multi_turn_loan_application() -> None:
    """Run a full loan-application conversation over one persistent session."""
    print("=== 💬 Multi-Turn Loan Application Workflow ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            advisor = Agent(
                client=client,
                name="LoanApplicationAdvisor",
                instructions="""You are a comprehensive Loan Application Advisor. Guide customers through:
                1. Understanding loan options
                2. Prequalification assessment
                3. Document requirements
                4. Next steps in the application process

                Remember the conversation context and reference earlier information when helpful.
                Be encouraging while being realistic about qualifications.""",
                tools=[get_loan_prequalification, get_required_documents],
            )

            # One session preserves context across every turn.
            session = advisor.create_session()
            conversation = [
                "Hi, I'm looking to buy my first home and need information about mortgage options.",
                "Can you check if I'd prequalify? My credit score is 680, annual income is $72,000, and I'm looking at homes around $280,000.",
                "Based on my prequalification status, what documents would I need for a conventional mortgage?",
                "Given my situation, would you recommend I improve my credit score first before applying?",
            ]

            for turn_number, message in enumerate(conversation, 1):
                print(f"\n📩 Turn {turn_number} - Customer: {message}")
                async with asyncio.timeout(90):
                    response = await advisor.run(message, session=session)
                assert response.text, f"The service returned no answer for turn {turn_number}."
                print(f"🏦 Advisor: {response.text}")
                print("-" * 50)
        finally:
            await client.client.close()
            await client.project_client.close()


## 🚀 Execute Multi-Turn Loan Application

In [ ]:
await multi_turn_loan_application()

## 📝 Key Takeaways

### Session Management in Applications

1. **Session Persistence**: A session maintains conversation context across multiple turns, essential for complex multi-turn discussions.

2. **Creating Sessions**: Start a session from the agent:
   ```python
   session = agent.create_session()
   ```

3. **Resuming Sessions**: After the first run the service assigns an id you can reconnect to:
   ```python
   resumed = agent.get_session(session.service_session_id)
   ```

4. **Running Turns**: Pass the session on every run to preserve context:
   ```python
   result = await agent.run(message, session=session)
   ```

### Agent Creation with Agent + FoundryChatClient

```python
with AzureCliCredential() as credential:
    client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
    agent = Agent(
        client=client,
        name="LoanApplicationAdvisor",
        instructions="...",
        tools=[get_loan_prequalification, get_required_documents],
    )
```

### Public APIs only

Agent Framework 1.17.0 exposes sessions and function tools directly. The earlier release-candidate
workaround (private `agent_framework._*` imports plus a `client_type` subclass that stripped tool schemas)
is no longer needed and has been removed.

### Use Case Benefits

- **Compliance**: Persistent sessions create audit trails for regulatory compliance
- **Customer Experience**: Customers don't need to repeat information
- **Context-Aware Responses**: Agents reference earlier parts of the conversation
- **Multi-Step Processes**: Ideal for loan applications, account onboarding, and consultations

### ⚠️ Disclaimer
This example uses simulated data for demonstration purposes. In production:
- Connect to actual backend systems
- Implement proper authentication and authorization
- Follow all regulatory requirements
- Ensure compliance for sensitive data
